# Import Packages

In [181]:
# Import Packages
%load_ext autoreload
%autoreload 2

import random

from dice_model import face_lookup, mcp_dice
from resolution import (
    apply_deterministic_mod,
    apply_reroll,
    effective_property,
)
from rules import (
    DiceModQuantity,
    PassiveRule,
    PropertyOverride,
    Window,
    pierce_rule,
    reroll_x_rule,
    reroll_any_rule,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Shadowcat vs The Black Widow

Shadowcat attacks The Black Widow with a 7 dice attack; The Black Widow has 4 defense dice.

## Special Rules for Shadowcat:

- During the attack, the defender does not add *Critical* results in its defense roll to its total successes and cannot add additional dice to the defense roll as a result of the *Critical* results.

In [182]:
# Passive Rules

hex = PassiveRule(
    name="Hex",
    override=PropertyOverride(
        face="Critical",
        property="additional_dice",
        effect=0
    )
)

crits_are_failure = PassiveRule(
    name="Critical is Failure",
    override=PropertyOverride(
        face="Critical",
        property="defense_value",
        effect=0
    )
)

martial_artist = PassiveRule(
 name="Martial Artist",
 override=PropertyOverride(
     face="Blank",
     property="defense_value",
     effect=1
 )   
)

widow_reroll_any = reroll_any_rule(
    name="Widow Reroll Any",
    window=Window.DEFENDER_MOD_SELF,
    temporary_overrides=[
        PropertyOverride(
            face="Skull",
            property="is_modifiable",
            effect=True
        )
    ]
)

# Create Dice Pools (Steps 4 & 5) 

In [183]:
# Step 4 - Create Attackers Dice Pool
attacker_pool = 7

# Step 5 - Create Defenders Dice Pool
defender_pool = 4

# Roll Initial Dice Pool (Steps 6 & 7)

In [184]:
# Step 6 - Roll the attacker's dice pool
attacker_original_roll = random.choices(mcp_dice, k=attacker_pool)

# Step 7 - Rolle the defenders' dice pool
defender_original_roll = random.choices(mcp_dice, k=defender_pool)

print(attacker_original_roll)
print(defender_original_roll)

['Shield', 'Hit', 'Wild', 'Wild', 'Critical', 'Blank', 'Wild']
['Wild', 'Hit', 'Wild', 'Skull']


# Resolve Criticals (Step 8)

In [185]:
attacker_additional_pool = sum(
    face_lookup[face].additional_dice
    for face in attacker_original_roll
)

attacker_additional_roll = random.choices(
    mcp_dice, 
    k=attacker_additional_pool
)

attacker_full_roll = attacker_original_roll + attacker_additional_roll

defender_additional_pool = sum(
    effective_property(
        face_name=face,
        property="additional_dice",
        passive_rules=[hex]
    )
    for face in defender_original_roll
)

defender_additional_roll = random.choices(
    mcp_dice, 
    k=defender_additional_pool
)

defender_full_roll = defender_original_roll + defender_additional_roll

# Attacker Modifies Self (Step 9.a.i)

In [186]:
attacker_current_roll = attacker_full_roll.copy()
attack_current_results = []
attacker_track_reroll = []

for result in attacker_current_roll:
    attack_current_results.append(
        face_lookup[result].attack_value
    )

# Defender Modify Self (Step 9.a.ii)

In [187]:
defender_current_roll = defender_full_roll.copy()
defend_current_results = []
defender_track_reroll = []

for result in defender_current_roll:
    defend_current_results.append(
        effective_property(
            face_name=result,
            property="defense_value",
            passive_rules=[martial_artist,crits_are_failure]
        )
    )

for rule in [widow_reroll_any]:
    defender_current_roll = apply_reroll(
        roll=defender_current_roll,
        results=defend_current_results,
        rule=rule,
        passive_rules=[]
        )

    defend_current_results = []

    for result in defender_current_roll:
        defend_current_results.append(
        effective_property(
            face_name=result,
            property="defense_value",
            passive_rules=[martial_artist,crits_are_failure]
        )
    )
    
    defender_track_reroll.append(defender_current_roll)

# Calculate Results After Self Mods

In [188]:
attacker_results = []
defender_results = []

for result in attacker_current_roll:
    attacker_results.append(
    face_lookup[result].attack_value
    )

for result in defender_current_roll:
    defender_results.append(
        effective_property(
            face_name=result,
            property="defense_value",
            passive_rules=[martial_artist,crits_are_failure]
        )
    )


# Attacker Mods Defender (Step 9.b.i)

# Calculate Success or Failure (Step 10)

In [189]:
attacker_results = []
defender_results = []

for result in attacker_current_roll:
    attacker_results.append(
    face_lookup[result].attack_value
    )

for result in defender_current_roll:
    defender_results.append(
        effective_property(
            face_name=result,
            property="defense_value",
            passive_rules=[martial_artist,crits_are_failure]
        )
    )

attack_successes = sum(attacker_results)
defend_succeses = sum(defender_results)
attack_damage = max(0,attack_successes-defend_succeses)

if attack_damage > 0:
    attack_result = f"Attack did {attack_damage} damage"
else:
    attack_result = "Attack did no damage"

In [190]:
print("Attacker Dice")
print(f"    Original roll:    {attacker_original_roll}")
print(f"    Exploded roll:    {attacker_additional_roll}")
print(f"    Full roll:        {attacker_full_roll}")
print(f"    Self mod roll:    {attacker_current_roll}")
print(f"    Reroll history:   {attacker_track_reroll}")

print("\n")

print("Defender Dice")
print(f"    Original roll:    {defender_original_roll}")
print(f"    Exploded roll:    {defender_additional_roll}")
print(f"    Full roll:        {defender_full_roll}")
print(f"    Self mod roll:    {defender_current_roll}")
print(f"    Reroll history:   {defender_track_reroll}")

print("\n")
print(f"Attacker successes: {sum(attacker_results)}")
print(f"Defender successes: {sum(defender_results)}")
print(attack_result)


Attacker Dice
    Original roll:    ['Shield', 'Hit', 'Wild', 'Wild', 'Critical', 'Blank', 'Wild']
    Exploded roll:    ['Hit']
    Full roll:        ['Shield', 'Hit', 'Wild', 'Wild', 'Critical', 'Blank', 'Wild', 'Hit']
    Self mod roll:    ['Shield', 'Hit', 'Wild', 'Wild', 'Critical', 'Blank', 'Wild', 'Hit']
    Reroll history:   []


Defender Dice
    Original roll:    ['Wild', 'Hit', 'Wild', 'Skull']
    Exploded roll:    []
    Full roll:        ['Wild', 'Hit', 'Wild', 'Skull']
    Self mod roll:    ['Wild', 'Shield', 'Wild', 'Skull']
    Reroll history:   [['Wild', 'Shield', 'Wild', 'Skull']]


Attacker successes: 6
Defender successes: 3
Attack did 3 damage
